> Check individual signals according to the community they belong.

In [1]:
import pathlib

import yaml
import pandas as pd

import jax
import jax.numpy as jnp

import trapallada.util
import trapallada.results.analysis
import trapallada.model.training

So that `import`ed Python modules are automatically reloaded when modified

In [2]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


So that *plotly*'s pictures are correctly rendered

In [3]:
import plotly.io as pio
pio.renderers.default = 'notebook'

# Parameters

Base directory

In [4]:
base_dir = trapallada.util.get_base_directory()
assert base_dir.exists()

Directory with all the files

In [5]:
res_dir = base_dir / 'results' / 'multichannel' / 'rates_large_infs' / 'time_z_5'
assert res_dir.exists()

*Sub*directory with the labels

In [6]:
annotations_dir = res_dir / 'annotations'
assert annotations_dir.exists()

The "selected by" file indicating the algorithm (VAE or AR) that selected every window.

In [7]:
selected_by_file = annotations_dir / 'signals_to_assess.parquet'
assert selected_by_file.exists()

The file with the annotations provided by Javi López Santiago.

In [8]:
annotations_file = annotations_dir / 'windows.xlsx'
assert annotations_file.exists()

Output file *merging* the above files

In [9]:
output_file = annotations_dir / 'annotated_windows.parquet'

# Input data

The *selected by* `DataFrame` stating the algorithm (VAE or AR) that selected every window.

In [10]:
selected_by_df = pd.read_parquet(selected_by_file)
selected_by_df

,start,end,tar file,source,signal,Selected by
14796,2004-02-29 23:34:36.224518,2004-03-01 05:17:59.744518,0201360201.tar,P0201360201PNX000SRCTSR8002.FTZ,0201360201/P0201360201PNX000SRCTSR8002,VAE
25125,2015-05-25 05:07:30.919731,2015-05-25 10:50:54.439731,0770180201.tar,P0770180201PNX000SRCTSR8001.FTZ,0770180201/P0770180201PNX000SRCTSR8001,AR
15937,2011-01-03 19:00:05.711776,2011-01-04 00:43:29.231776,0652330101.tar,P0652330101PNX000SRCTSR8001.FTZ,0652330101/P0652330101PNX000SRCTSR8001,VAE
22425,2015-03-26 05:54:06.667239,2015-03-26 11:37:30.187239,0764800301.tar,P0764800301PNX000SRCTSR8005.FTZ,0764800301/P0764800301PNX000SRCTSR8005,NaN
29039,2015-06-27 04:31:17.762896,2015-06-27 10:14:41.282896,0761970101.tar,P0761970101PNX000SRCTSR8002.FTZ,0761970101/P0761970101PNX000SRCTSR8002,NaN
...,...,...,...,...,...,...
27106,2014-10-04 23:21:51.077119,2014-10-05 05:05:14.597119,0728990101.tar,P0728990101PNX000SRCTSR8001.FTZ,0728990101/P0728990101PNX000SRCTSR8001,AR
9253,2008-06-27 08:47:46.841629,2008-06-27 14:31:10.361629,0551750301.tar,P0551750301PNX000SRCTSR8001.FTZ,0551750301/P0551750301PNX000SRCTSR8001,AR
15049,2017-10-24 15:17:34.836966,2017-10-24 21:00:58.356966,0795730101.tar,P0795730101PNX000SRCTSR8001.FTZ,0795730101/P0795730101PNX000SRCTSR8001,VAE
5371,2013-09-27 21:06:30.736658,2013-09-28 02:49:54.256658,0724380101.tar,P0724380101PNX000SRCTSR8003.FTZ,0724380101/P0724380101PNX000SRCTSR8003,NaN


The `DataFrame` with the labels provided by the annotator. Its index (window *id*) must be the same as that of the above `DataFrame`.

In [11]:
# labels_df = pd.read_excel(labels_file)
labels_df = pd.read_excel(annotations_file)
labels_df = labels_df.drop(columns=labels_df.columns[0])
labels_df.index = selected_by_df.index
labels_df

,Label,Type,Subtype
14796,Interesting,doubleExponential,flare
25125,NaN,NaN,NaN
15937,Interesting,doubleExponential,flare
22425,Interesting,background,NaN
29039,NOT interesting,NaN,NaN
...,...,...,...
27106,NOT interesting,NaN,NaN
9253,NOT interesting,NaN,NaN
15049,Interesting,background,NaN
5371,NOT interesting,NaN,NaN


They are merged together

In [12]:
res_df = pd.concat((selected_by_df, labels_df), axis=1)
res_df

,start,end,tar file,source,signal,Selected by,Label,Type,Subtype
14796,2004-02-29 23:34:36.224518,2004-03-01 05:17:59.744518,0201360201.tar,P0201360201PNX000SRCTSR8002.FTZ,0201360201/P0201360201PNX000SRCTSR8002,VAE,Interesting,doubleExponential,flare
25125,2015-05-25 05:07:30.919731,2015-05-25 10:50:54.439731,0770180201.tar,P0770180201PNX000SRCTSR8001.FTZ,0770180201/P0770180201PNX000SRCTSR8001,AR,NaN,NaN,NaN
15937,2011-01-03 19:00:05.711776,2011-01-04 00:43:29.231776,0652330101.tar,P0652330101PNX000SRCTSR8001.FTZ,0652330101/P0652330101PNX000SRCTSR8001,VAE,Interesting,doubleExponential,flare
22425,2015-03-26 05:54:06.667239,2015-03-26 11:37:30.187239,0764800301.tar,P0764800301PNX000SRCTSR8005.FTZ,0764800301/P0764800301PNX000SRCTSR8005,NaN,Interesting,background,NaN
29039,2015-06-27 04:31:17.762896,2015-06-27 10:14:41.282896,0761970101.tar,P0761970101PNX000SRCTSR8002.FTZ,0761970101/P0761970101PNX000SRCTSR8002,NaN,NOT interesting,NaN,NaN
...,...,...,...,...,...,...,...,...,...
27106,2014-10-04 23:21:51.077119,2014-10-05 05:05:14.597119,0728990101.tar,P0728990101PNX000SRCTSR8001.FTZ,0728990101/P0728990101PNX000SRCTSR8001,AR,NOT interesting,NaN,NaN
9253,2008-06-27 08:47:46.841629,2008-06-27 14:31:10.361629,0551750301.tar,P0551750301PNX000SRCTSR8001.FTZ,0551750301/P0551750301PNX000SRCTSR8001,AR,NOT interesting,NaN,NaN
15049,2017-10-24 15:17:34.836966,2017-10-24 21:00:58.356966,0795730101.tar,P0795730101PNX000SRCTSR8001.FTZ,0795730101/P0795730101PNX000SRCTSR8001,VAE,Interesting,background,NaN
5371,2013-09-27 21:06:30.736658,2013-09-28 02:49:54.256658,0724380101.tar,P0724380101PNX000SRCTSR8003.FTZ,0724380101/P0724380101PNX000SRCTSR8003,NaN,NOT interesting,NaN,NaN


It is saved

In [13]:
res_df.to_parquet(output_file, index=True)

# Precision

The *sub*`DataFrame`s for VAE...

In [14]:
vae_selected_df = res_df[res_df['Selected by'] == 'VAE'].drop(columns=['Selected by'])

, AR...

In [15]:
ar_selected_df = res_df[res_df['Selected by'] == 'AR'].drop(columns=['Selected by'])

...and *random*

In [16]:
random_selected_df = res_df[pd.isna(res_df['Selected by'])].drop(columns=['Selected by'])
random_selected_df.shape

(82, 8)

Another *sub*`DataFrame` with those that were annotated as `Interesting`

In [17]:
interesting_df = res_df[res_df['Label'] == 'Interesting']
interesting_df.shape

(125, 9)

# Precision

In [47]:
overall_precision = ((res_df['Label'] == 'Interesting') & (res_df['Type'] != 'background')).sum().item() / len(res_df)
overall_precision

0.425

## VAE

In [18]:
# vae_precision = (vae_selected_df['Label'] == 'Interesting').sum().item() / len(vae_selected_df)
vae_precision = ((vae_selected_df['Label'] == 'Interesting') & (vae_selected_df['Type'] != 'background')).sum().item() / len(vae_selected_df)
vae_precision

0.7166666666666667

## AR

In [19]:
# ar_precision = (ar_selected_df['Label'] == 'Interesting').sum().item() / len(ar_selected_df)
ar_precision = ((ar_selected_df['Label'] == 'Interesting') & (ar_selected_df['Type'] != 'background')).sum().item() / len(ar_selected_df)
ar_precision

0.39655172413793105

## Random

In [20]:
# random_precision = (random_selected_df['Label'] == 'Interesting').sum().item() / len(random_selected_df)
random_precision = ((random_selected_df['Label'] == 'Interesting') & (random_selected_df['Type'] != 'background')).sum().item() / len(random_selected_df)
random_precision

0.23170731707317074

# Recall

It requires a probability/score for every point...

In [21]:
# vae_recall = interesting_df.index.isin(vae_selected_df.index).sum().item()
# vae_recall

# Distances

In [22]:
selected_df = pd.read_excel(res_dir / 'selected.xlsx')
selected_df

,# window,start,end,tar file,source,signal
0,8248,2014-06-03 10:18:04.165,2014-06-03 16:01:27.685,0740290301.tar,P0740290301PNX000SRCTSR8002.FTZ,0740290301/P0740290301PNX000SRCTSR8002
1,12392,2009-06-03 15:06:00.097,2009-06-03 20:49:23.617,0605950101.tar,P0605950101PNX000SRCTSR8001.FTZ,0605950101/P0605950101PNX000SRCTSR8001
2,16827,2014-04-13 18:02:03.317,2014-04-13 23:45:26.837,0720173901.tar,P0720173901PNX000SRCTSR8001.FTZ,0720173901/P0720173901PNX000SRCTSR8001
3,26795,2012-10-23 22:03:27.911,2012-10-24 03:46:51.431,0690744401.tar,P0690744401PNX000SRCTSR8002.FTZ,0690744401/P0690744401PNX000SRCTSR8002


In [23]:
i_selected_windows = selected_df['# window'].values
i_selected_windows

array([ 8248, 12392, 16827, 26795])

## VAE

In [24]:
vae_df = pd.read_parquet(res_dir / 'vae.parquet')
vae_df.head()

,start,end,tar file,source,z_sd_1,z_sd_2,z_sd_3,z_sd_4,z_sd_5,wasserstein cluster in z,...,cluster in projected space,RA,DEC,CRA,CDEC,main_type,other_types,main_id,# window,signal
0,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNX000SRCTSR8001.FTZ,0.028998,0.030307,0.029041,0.031015,0.038947,4,...,1,201.089746,31.447543,201.090309,31.447292,QSO,*|Opt|QSO|Rad,SDSS J132421.64+312650.8,0,0821871301/P0821871301PNX000SRCTSR8001
1,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNX000SRCTSR8002.FTZ,0.028151,0.027799,0.026894,0.022741,0.036637,4,...,1,200.941274,31.658391,200.941912,31.658179,QSO,Opt|QSO,SDSS J132346.07+313929.7,0,0821871301/P0821871301PNX000SRCTSR8002
2,2018-06-08 20:30:01.369583,2018-06-09 02:13:24.889583,0821871301.tar,P0821871301PNX000SRCTSR8001.FTZ,0.024245,0.028545,0.026479,0.027166,0.031849,4,...,1,201.089746,31.447543,201.090309,31.447292,QSO,*|Opt|QSO|Rad,SDSS J132421.64+312650.8,1,0821871301/P0821871301PNX000SRCTSR8001
3,2018-06-08 20:30:01.369583,2018-06-09 02:13:24.889583,0821871301.tar,P0821871301PNX000SRCTSR8002.FTZ,0.030794,0.029051,0.026218,0.024145,0.037635,4,...,1,200.941274,31.658391,200.941912,31.658179,QSO,Opt|QSO,SDSS J132346.07+313929.7,1,0821871301/P0821871301PNX000SRCTSR8002
4,2018-06-08 20:45:21.169583,2018-06-09 02:28:44.689583,0821871301.tar,P0821871301PNX000SRCTSR8001.FTZ,0.030508,0.029943,0.030053,0.029959,0.039443,4,...,1,201.089746,31.447543,201.090309,31.447292,QSO,*|Opt|QSO|Rad,SDSS J132421.64+312650.8,2,0821871301/P0821871301PNX000SRCTSR8001


In [25]:
z_mean = vae_df.filter(regex=r'z_\d+').values
z_sd = vae_df.filter(regex=r'z_sd_\d+').values
z_mean.shape, z_sd.shape

((30856, 5), (30856, 5))

In [26]:
vae_distances = jax.vmap(
    trapallada.results.analysis.wasserstein_between_one_and_many_gaussians, in_axes=(0, 0, None, None)
    )(
        z_mean[i_selected_windows],
        z_sd[i_selected_windows],
        z_mean, z_sd
)
# assert (vae_distances[jnp.arange(len(i_selected_windows)), i_selected_windows] == 0.0).all().item()

In [27]:
vae_rank = vae_distances.min(axis=0)
assert (vae_rank[i_selected_windows] == 0.0).all().item()

## AR

In [28]:
with (res_dir / 'ar_model.txt').open('r') as f:
    ar_file = f.readline()

In [29]:
with (res_dir / 'ar_model.txt').open('r') as f:
    ar_file = f.readline()
ar_df = pd.read_parquet(ar_file)
ar_df.head()

,start,end,tar file,source,z_1,z_2,z_3,z_4,z_5,z_6,...,cluster in projected space,RA,DEC,CRA,CDEC,main_type,other_types,main_id,# window,signal
0,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNX000SRCTSR8001.FTZ,-0.069514,0.069419,0.066852,-0.031498,0.041259,0.029128,...,1,201.089746,31.447543,201.090309,31.447292,QSO,*|Opt|QSO|Rad,SDSS J132421.64+312650.8,0,0821871301/P0821871301PNX000SRCTSR8001
1,2018-06-08 20:14:41.569583,2018-06-09 01:58:05.089583,0821871301.tar,P0821871301PNX000SRCTSR8002.FTZ,0.048524,0.060231,0.120665,0.151738,-0.024934,0.019107,...,1,200.941274,31.658391,200.941912,31.658179,QSO,Opt|QSO,SDSS J132346.07+313929.7,0,0821871301/P0821871301PNX000SRCTSR8002
2,2018-06-08 20:30:01.369583,2018-06-09 02:13:24.889583,0821871301.tar,P0821871301PNX000SRCTSR8001.FTZ,-0.040494,0.042494,0.067655,-0.041173,-0.012557,0.031347,...,1,201.089746,31.447543,201.090309,31.447292,QSO,*|Opt|QSO|Rad,SDSS J132421.64+312650.8,1,0821871301/P0821871301PNX000SRCTSR8001
3,2018-06-08 20:30:01.369583,2018-06-09 02:13:24.889583,0821871301.tar,P0821871301PNX000SRCTSR8002.FTZ,0.042855,0.034260,0.112990,0.159840,-0.036125,0.001929,...,1,200.941274,31.658391,200.941912,31.658179,QSO,Opt|QSO,SDSS J132346.07+313929.7,1,0821871301/P0821871301PNX000SRCTSR8002
4,2018-06-08 20:45:21.169583,2018-06-09 02:28:44.689583,0821871301.tar,P0821871301PNX000SRCTSR8001.FTZ,-0.042048,0.052945,0.052545,-0.030169,-0.020930,-0.002064,...,1,201.089746,31.447543,201.090309,31.447292,QSO,*|Opt|QSO|Rad,SDSS J132421.64+312650.8,2,0821871301/P0821871301PNX000SRCTSR8001


In [30]:
z_mean = ar_df.filter(regex=r'z_\d+').values
z_mean.shape

(30856, 125)

In [31]:
def euclidean_distance_between_one_and_many(x, Y):
    return jnp.sqrt(jnp.sum((x[None, :] - Y) ** 2, axis=-1))
euclidean_distance_between_one_and_many(z_mean[0], z_mean)

Array([0.       , 1.0767276, 0.288939 , ..., 1.122915 , 1.1408046,
       1.1129788], dtype=float32)

In [32]:
ar_distances = jax.vmap(
    euclidean_distance_between_one_and_many, in_axes=(0, None)
    )(
        z_mean[i_selected_windows],
        z_mean
)
# assert (ar_distances[jnp.arange(len(i_selected_windows)), i_selected_windows] == 0.0).all().item()

In [33]:
ar_rank = ar_distances.min(axis=0)
assert (ar_rank[i_selected_windows] == 0.0).all().item()

In [34]:
ar_rank

Array([1.145758 , 1.1216351, 1.1474112, ..., 1.2224596, 1.171832 ,
       1.2021964], dtype=float32)

# Latent space

In [35]:
from typing import Callable

import torch
from torch.utils.data import DataLoader

import jax.numpy as jnp
import jax.random as jr
import equinox as eqx

from jaxtyping import PRNGKeyArray

In [36]:
ds, positions_df, model, training_df, parameters = trapallada.results.analysis.load_dir(res_dir)

In [48]:
batch_size = 128
dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

In [ ]:
def loss(
        model: eqx.Module,
        x: Float[Array, 'batch channel length'],
        mask: Float[Array, 'batch 1 length'],
        x_sd: float,
        mask_non_gti_intervals: bool,
        key: PRNGKeyArray
        ) -> tuple[Float[Array, ''], tuple[Float[Array, ''], Float[Array, '']]]:

    # for the sake of readability
    batch_size = len(x)
    overall_times = jnp.array(mask.shape).prod()
    overall_good_times = mask.sum()

    x_pred, z_mean, z_sd = jax.vmap(model)(x, jr.split(key, batch_size))

    # if this is a Conditional VAE...
    if isinstance(model, trapallada.model.nn.CVAE):
        
        # ...it is assumed that the last channel is the condition, and hence must not be reconstructed
        x = x.at[:, :-1, :].get()

    assert mask.ndim == 2

    # reconstruction term
    if mask_non_gti_intervals:
        loglikelihood = (dist.Normal(loc=x_pred, scale=x_sd).log_prob(x) * mask[:, None, :]).sum() / overall_good_times
    else:
        loglikelihood = dist.Normal(loc=x_pred, scale=x_sd).log_prob(x).sum() / overall_times

    # KL divergence (notice the mask cannot be applied here because it's on the time domain)
    # if mask_non_gti_intervals:
    #     kl = -0.5 * jnp.sum(1. + 2.*jnp.log(z_sd) - z_mean**2 - z_sd**2) / overall_good_times
    # else:
    #     kl = -0.5 * jnp.sum(1. + 2.*jnp.log(z_sd) - z_mean**2 - z_sd**2) / batch_size
    kl = -0.5 * jnp.sum(1. + 2.*jnp.log(z_sd) - z_mean**2 - z_sd**2) / batch_size

    return -loglikelihood + kl, (loglikelihood, kl)

In [ ]:
def loss_metrics(
        model: eqx.Module, dataloader: torch.utils.data.DataLoader, loss_function: Callable,
        x_sd: float, mask_non_gti_intervals: bool, key: PRNGKeyArray):

    log_likelihood, kl_divergence = [], []

    for _, x, mask in dataloader:

        key, subkey = jr.split(key)

        x = x.numpy()
        mask = mask.numpy()

        x_pred, z_mean, z_sd = jax.vmap(model)(x, jr.split(key, batch_size))
    
        _, (log_likelihood_value, kl_value) = loss_function(model, x, mask, x_sd, mask_non_gti_intervals, subkey)

        log_likelihood.append(log_likelihood_value)
        kl_divergence.append(kl_value)

    return log_likelihood, kl_divergence

In [40]:
log_likelihood, kl_divergence = loss_metrics(
    model,
    dl,
    trapallada.model.training.loss,
    parameters['loss']['x_sd'],
    parameters['training']['mask_non_gti_intervals'],
    jr.PRNGKey(parameters['PRNG seeds']['sampling']['validation set']),
)

In [45]:
log_likelihood[0]

Array(-88.744125, dtype=float32)

In [43]:
len(log_likelihood), len(kl_divergence)

(242, 242)

In [41]:
pd.DataFrame({'log_likelihood': log_likelihood, 'kl_divergence': kl_divergence})

,log_likelihood,kl_divergence
0,-88.744125,0.07130009
1,-89.634224,0.09213896
2,-90.400116,0.08876752
3,-79.151215,0.108175084
4,-75.86151,0.09424692
...,...,...
237,-53.816563,0.077180214
238,-66.00412,0.0852928
239,-44.54709,0.108749345
240,-72.1689,0.08034205
